In [ ]:
# Cell 1: Check GPU availability
!nvidia-smi

In [ ]:
# Cell 2: Install required packages
print("📦 Installing dependencies...")
!pip install -q vllm datasets pandas numpy scikit-learn matplotlib seaborn tqdm
print("✅ Dependencies installed!")

In [ ]:
# Cell 3: CENTRALIZED CONFIGURATION - SINGLE SOURCE OF TRUTH
# ============================================================================
# 🎯 Change all evaluation parameters HERE ONLY!
# All other cells dynamically reference this configuration.
# ============================================================================

import os

CONFIG = {
    # ============================================================
    # MODEL CONFIGURATION
    # ============================================================
    "model_name": "Qwen/Qwen2.5-1.5B-Instruct",   # Base model — no LoRA adapters

    # ============================================================
    # PATH CONFIGURATION
    # ============================================================
    "data_path": "/kaggle/input/datasets/abirashab/train-test-val",
    "test_file": "test.jsonl",

    # ============================================================
    # EVALUATION SETTINGS
    # ============================================================
    "eval_limit": 2000,          # None = full dataset, or set a number (e.g., 100, 500)
    "suspicious_ratio": 0.30,    # Fraction of suspicious samples when eval_limit is set
    "prioritize_shortest": True, # Take shortest logs first when sampling
    "max_new_tokens": 512,       # Maximum new tokens to generate per response
    "temperature": 0.7,          # Sampling temperature (matches fine-tuned evaluation)
    "top_p": 0.9,                # Nucleus sampling parameter

    # ============================================================
    # PROMPT CONFIGURATION
    # ============================================================
    "max_input_chars": 6000,     # Truncate input JSON at this many characters

    # ============================================================
    # vLLM SETTINGS
    # ============================================================
    # Qwen2.5-1.5B requires ~3 GB — single T4 is sufficient.
    # Set tensor_parallel_size=2 if you want to use both Kaggle T4s.
    "dtype": "float16",
    "gpu_memory_utilization": 0.90,
    "tensor_parallel_size": 1,

    # ============================================================
    # OUTPUT SETTINGS
    # ============================================================
    "show_examples": 10,
    "save_results": True,
    "results_file": "base_model_evaluation_results.csv",
    "metrics_file": "base_model_metrics_summary.csv",
}

# Auto-derived paths
CONFIG["test_file_path"] = f"{CONFIG['data_path']}/{CONFIG['test_file']}"

# ============================================================
# DISPLAY CONFIGURATION
# ============================================================
print("=" * 80)
print("📋 BASE MODEL EVALUATION CONFIGURATION")
print("=" * 80)

print(f"\n🤖 Model:")
print(f"   {CONFIG['model_name']}  (base — no fine-tuning)")

print(f"\n📂 Paths:")
print(f"   Test data: {CONFIG['test_file_path']}")

print(f"\n🎯 Evaluation Settings:")
if CONFIG["eval_limit"] is None:
    print(f"   Eval limit: Full dataset (all examples)")
else:
    print(f"   Eval limit: {CONFIG['eval_limit']:,} examples")
    print(f"   Suspicious ratio: {CONFIG['suspicious_ratio'] * 100:.0f}%")
    print(f"   Prioritize shortest: {CONFIG['prioritize_shortest']}")
print(f"   Max new tokens:  {CONFIG['max_new_tokens']}")
print(f"   Temperature:     {CONFIG['temperature']}")
print(f"   Top-p:           {CONFIG['top_p']}")

print(f"\n⚡ vLLM Settings:")
print(f"   dtype:                  {CONFIG['dtype']}")
print(f"   GPU memory utilization: {CONFIG['gpu_memory_utilization']}")
print(f"   Tensor parallel size:   {CONFIG['tensor_parallel_size']}")

print(f"\n📊 Prompt Settings:")
print(f"   Max input chars: {CONFIG['max_input_chars']:,}")

print(f"\n💾 Output Settings:")
print(f"   Show examples: {CONFIG['show_examples']}")
print(f"   Save results:  {CONFIG['save_results']}")
if CONFIG["save_results"]:
    print(f"   Results file:  {CONFIG['results_file']}")
    print(f"   Metrics file:  {CONFIG['metrics_file']}")

if os.path.exists(CONFIG["test_file_path"]):
    print(f"\n✅ Test file found")
else:
    print(f"\n❌ Test file not found at {CONFIG['test_file_path']}")
    print("   Please add the train-test-val dataset to Kaggle")

print("=" * 80)
print("✅ Configuration loaded!")
print("=" * 80)

In [ ]:
# Cell 4: Load model with vLLM (offline batch inference)
# vLLM's LLM class downloads the model once, caches it to ~/.cache/huggingface,
# and handles batching + PagedAttention automatically.
print("🔄 Loading base model with vLLM...\n")

from vllm import LLM, SamplingParams

llm = LLM(
    model=CONFIG["model_name"],
    dtype=CONFIG["dtype"],
    trust_remote_code=True,
    gpu_memory_utilization=CONFIG["gpu_memory_utilization"],
    tensor_parallel_size=CONFIG["tensor_parallel_size"],
    # Disable prefix caching for first run to avoid potential cache issues
    enable_prefix_caching=False,
)

sampling_params = SamplingParams(
    temperature=CONFIG["temperature"],
    top_p=CONFIG["top_p"],
    max_tokens=CONFIG["max_new_tokens"],
)

print(f"\n✅ Base model loaded: {CONFIG['model_name']}")
print(f"⚡ Inference engine: vLLM (offline batch mode)")
print(f"   All {CONFIG['eval_limit'] or 'all'} prompts will be submitted as a single batch.")

In [ ]:
# Cell 5: Load test dataset
print("🔄 Loading test dataset...\n")

from datasets import load_dataset
import json

test_dataset = load_dataset(
    "json",
    data_files={"test": CONFIG["test_file_path"]},
)["test"]

print(f"✅ Test dataset loaded: {len(test_dataset):,} examples")
print(f"\n📋 Dataset columns: {test_dataset.column_names}")

print(f"\n" + "=" * 80)
print("📋 SAMPLE TEST ENTRY")
print("=" * 80)

sample = test_dataset[0]

print(f"\n1️⃣  INSTRUCTION:")
print(f"   {sample['instruction']}")

print(f"\n2️⃣  INPUT (first 300 chars):")
print(f"   {sample['input'][:300]}...")

print(f"\n3️⃣  EXPECTED OUTPUT (first 200 chars):")
print(f"   {sample['output'][:200]}...")
print("=" * 80)

In [ ]:
# Cell 6: Utility functions
print("🔄 Defining utility functions...\n")

import re


def build_prompt(instruction: str, input_text: str) -> str:
    """
    Build the evaluation prompt using the SAME format as training data.

    Format: instruction + "\n\n" + input + "\n\nAnalysis:\n"
    This is identical to the prompt used in the fine-tuned model evaluation
    so results are directly comparable.
    """
    if len(input_text) > CONFIG["max_input_chars"]:
        input_text = input_text[: CONFIG["max_input_chars"]] + "... [truncated]"
    return f"{instruction}\n\n{input_text}\n\nAnalysis:\n"


def extract_status_label(text: str) -> str:
    """
    Extract NORMAL / SUSPICIOUS / UNKNOWN from model output.

    Checks for the training-data output format first, then falls back
    to the older plain-text format.
    """
    if not text or not isinstance(text, str):
        return "UNKNOWN"

    text_lower = text.lower()

    suspicious_markers = [
        "security alert",
        "**security alert",
        "suspicious activity",
        "malicious activity",
        "attack detected",
        "threat detected",
    ]
    normal_markers = [
        "normal activity detected",
        "**normal activity",
        "no malicious",
        "no suspicious indicators",
        "standard activity",
        "routine activity",
    ]

    # Check suspicious first (stronger signal)
    for marker in suspicious_markers:
        if marker in text_lower:
            return "SUSPICIOUS"

    for marker in normal_markers:
        if marker in text_lower:
            return "NORMAL"

    # Fallback: plain-text format (Status: Normal / Status: Suspicious)
    if "status: normal" in text_lower or "status:normal" in text_lower:
        return "NORMAL"
    if "status: suspicious" in text_lower or "status:suspicious" in text_lower:
        return "SUSPICIOUS"

    return "UNKNOWN"


def extract_technique_id(text: str):
    """Extract the first MITRE technique ID from text (e.g., T1234 or T1234.001)."""
    match = re.search(r"T\d{4}(?:\.\d{3})?", text.upper())
    return match.group(0) if match else None


def calculate_exact_match(pred: str, target: str) -> float:
    return 1.0 if pred.strip().lower() == target.strip().lower() else 0.0


def calculate_partial_match(pred: str, target: str) -> float:
    pred_lower = pred.strip().lower()
    target_lower = target.strip().lower()
    target_words = set(target_lower.split())
    pred_words = set(pred_lower.split())
    if not target_words:
        return 0.0
    overlap = len(target_words.intersection(pred_words))
    return overlap / len(target_words)


def calculate_f1_score(pred: str, target: str) -> float:
    pred_words = set(pred.strip().lower().split())
    target_words = set(target.strip().lower().split())
    if not pred_words or not target_words:
        return 0.0
    overlap = len(pred_words.intersection(target_words))
    precision = overlap / len(pred_words)
    recall = overlap / len(target_words)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


print("✅ Utility functions defined:")
print("   - build_prompt()             — formats prompt identical to training data")
print("   - extract_status_label()     — parse NORMAL / SUSPICIOUS / UNKNOWN")
print("   - extract_technique_id()     — extract MITRE technique IDs")
print("   - calculate_exact_match()    — binary string match")
print("   - calculate_partial_match()  — word-overlap ratio")
print("   - calculate_f1_score()       — word-level F1")
print("\n" + "=" * 80)

In [ ]:
# Cell 7: Stratified sampling + build prompt list
print("🔄 Preparing evaluation samples...\n")

from collections import Counter

# ── Stratified sampling (mirrors fine-tuned metrics notebook exactly) ──────
if CONFIG["eval_limit"] is None:
    eval_samples = test_dataset
    print(f"Evaluating on FULL test set: {len(eval_samples):,} examples")
else:
    print(f"📊 Stratified sampling: {CONFIG['suspicious_ratio'] * 100:.0f}% suspicious, "
          f"{(1 - CONFIG['suspicious_ratio']) * 100:.0f}% normal")

    indexed_data = [
        {
            "index": i,
            "status": extract_status_label(test_dataset[i]["output"]),
            "length": len(test_dataset[i]["input"]),
        }
        for i in range(len(test_dataset))
    ]

    suspicious_samples = [s for s in indexed_data if s["status"] == "SUSPICIOUS"]
    normal_samples     = [s for s in indexed_data if s["status"] == "NORMAL"]
    unknown_samples    = [s for s in indexed_data if s["status"] == "UNKNOWN"]

    print(f"   Available: {len(suspicious_samples)} suspicious, "
          f"{len(normal_samples)} normal, {len(unknown_samples)} unknown")

    if len(suspicious_samples) == 0 and len(normal_samples) == 0:
        raise ValueError(
            "Status extraction failed — check test data format.\n"
            f"Sample output: {test_dataset[0]['output'][:300]}"
        )

    if CONFIG["prioritize_shortest"]:
        suspicious_samples.sort(key=lambda x: x["length"])
        normal_samples.sort(key=lambda x: x["length"])
        print("   Prioritizing shortest logs")

    target_suspicious = int(CONFIG["eval_limit"] * CONFIG["suspicious_ratio"])
    target_normal     = CONFIG["eval_limit"] - target_suspicious

    selected_suspicious = suspicious_samples[: min(target_suspicious, len(suspicious_samples))]
    selected_normal     = normal_samples[: min(target_normal, len(normal_samples))]

    selected_indices = [s["index"] for s in selected_suspicious + selected_normal]

    print(f"   Selected: {len(selected_suspicious)} suspicious, {len(selected_normal)} normal")

    if CONFIG["prioritize_shortest"] and selected_suspicious:
        print(f"   Suspicious lengths: "
              f"{min(s['length'] for s in selected_suspicious):,} – "
              f"{max(s['length'] for s in selected_suspicious):,} chars")
    if CONFIG["prioritize_shortest"] and selected_normal:
        print(f"   Normal lengths:     "
              f"{min(s['length'] for s in selected_normal):,} – "
              f"{max(s['length'] for s in selected_normal):,} chars")

    eval_samples = test_dataset.select(selected_indices)
    print(f"\n✅ Evaluating on {len(eval_samples):,} stratified samples "
          f"(out of {len(test_dataset):,})")

# ── Build prompt list ───────────────────────────────────────────────────────
prompts = [
    build_prompt(ex["instruction"], ex["input"])
    for ex in eval_samples
]
expected_outputs = [ex["output"] for ex in eval_samples]

print(f"\n✅ {len(prompts):,} prompts built and ready for batch inference.")
print(f"\nSample prompt (first 400 chars):")
print(prompts[0][:400] + "...")

In [ ]:
# Cell 8: Run batch inference with vLLM
# vLLM submits ALL prompts at once and processes them with continuous batching,
# which is significantly faster than a sequential transformers loop.
print("🚀 Running batch inference with vLLM...\n")
print(f"   Submitting {len(prompts):,} prompts in a single batch.")
print(f"   Temperature: {CONFIG['temperature']}, Top-p: {CONFIG['top_p']}, "
      f"Max new tokens: {CONFIG['max_new_tokens']}\n")

import time
import numpy as np

start_time = time.time()

vllm_outputs = llm.generate(prompts, sampling_params)

elapsed = time.time() - start_time

# Extract generated text from each RequestOutput
predictions = [out.outputs[0].text.strip() for out in vllm_outputs]

print(f"\n✅ Batch inference complete in {elapsed / 60:.2f} minutes "
      f"({elapsed / len(prompts):.2f} sec/example)")

# ── Compute per-example metrics ─────────────────────────────────────────────
results = []
exact_matches = 0
partial_match_scores = []
f1_scores = []

for i, (prediction, expected) in enumerate(zip(predictions, expected_outputs)):
    exact_match   = calculate_exact_match(prediction, expected)
    partial_match = calculate_partial_match(prediction, expected)
    f1            = calculate_f1_score(prediction, expected)

    exact_matches += exact_match
    partial_match_scores.append(partial_match)
    f1_scores.append(f1)

    results.append({
        "index":         i,
        "instruction":   eval_samples[i]["instruction"],
        "input":         eval_samples[i]["input"],
        "expected":      expected,
        "predicted":     prediction,
        "exact_match":   exact_match,
        "partial_match": partial_match,
        "f1_score":      f1,
    })

# Show example predictions
show_n = min(CONFIG["show_examples"], len(results))
for i in range(show_n):
    r = results[i]
    print(f"\n{'=' * 80}")
    print(f"Example {i + 1}:")
    print(f"Input:     {r['input'][:80]}...")
    print(f"Expected:  {r['expected'][:200]}...")
    print(f"Predicted: {r['predicted'][:200]}...")
    print(f"Metrics:   Exact={r['exact_match']}, "
          f"Partial={r['partial_match']:.2f}, F1={r['f1_score']:.2f}")
    print(f"{'=' * 80}")

In [ ]:
# Cell 9: Basic statistics
print("\n" + "=" * 80)
print("📊 BASIC EVALUATION STATISTICS")
print("=" * 80 + "\n")

import numpy as np

input_lengths = [len(r["input"]) for r in results]

print(f"Total examples processed: {len(results):,}")
print(f"Inference time:           {elapsed / 60:.2f} minutes")
print(f"Time per sample:          {elapsed / len(results):.2f} seconds")
print(f"\n📈 Input Length Statistics (characters):")
print(f"   Min length:     {min(input_lengths):,}")
print(f"   Max length:     {max(input_lengths):,}")
print(f"   Average length: {np.mean(input_lengths):,.0f}")
print(f"   Median length:  {np.median(input_lengths):,.0f}")
print(f"   Std deviation:  {np.std(input_lengths):,.0f}")

In [ ]:
# Cell 10: Calculate comprehensive metrics
print("\n" + "=" * 80)
print("📊 CALCULATING COMPREHENSIVE METRICS")
print("=" * 80 + "\n")

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
import pandas as pd
from collections import Counter

y_true_status = [extract_status_label(r["expected"])  for r in results]
y_pred_status = [extract_status_label(r["predicted"]) for r in results]

print("=" * 80)
print("🎯 STATUS CLASSIFICATION METRICS (Normal vs Suspicious)")
print("=" * 80 + "\n")

unique_status_labels = sorted(list(set(y_true_status + y_pred_status)))
print(f"📋 Status labels found: {unique_status_labels}\n")

print(f"Expected label distribution:")
print(f"  {Counter(y_true_status)}\n")
print(f"Predicted label distribution:")
print(f"  {Counter(y_pred_status)}\n")

status_accuracy           = accuracy_score(y_true_status, y_pred_status)
status_precision_macro    = precision_score(y_true_status, y_pred_status, average="macro",    zero_division=0)
status_precision_weighted = precision_score(y_true_status, y_pred_status, average="weighted", zero_division=0)
status_recall_macro       = recall_score(   y_true_status, y_pred_status, average="macro",    zero_division=0)
status_recall_weighted    = recall_score(   y_true_status, y_pred_status, average="weighted", zero_division=0)
status_f1_macro           = f1_score(       y_true_status, y_pred_status, average="macro",    zero_division=0)
status_f1_weighted        = f1_score(       y_true_status, y_pred_status, average="weighted", zero_division=0)

print("🎯 STATUS CLASSIFICATION OVERALL METRICS:")
print(f"   Accuracy:             {status_accuracy:.4f} ({status_accuracy * 100:.2f}%)")
print(f"\n   Precision (Macro):    {status_precision_macro:.4f}")
print(f"   Precision (Weighted): {status_precision_weighted:.4f}")
print(f"\n   Recall (Macro):       {status_recall_macro:.4f}")
print(f"   Recall (Weighted):    {status_recall_weighted:.4f}")
print(f"\n   F1-Score (Macro):     {status_f1_macro:.4f}")
print(f"   F1-Score (Weighted):  {status_f1_weighted:.4f}")

print(f"\n📊 DETAILED STATUS CLASSIFICATION REPORT:")
print(classification_report(y_true_status, y_pred_status, zero_division=0))

status_conf_matrix = confusion_matrix(
    y_true_status, y_pred_status, labels=unique_status_labels
)

# Word-level metrics
avg_partial_match   = np.mean(partial_match_scores)
avg_f1_word         = np.mean(f1_scores)
exact_match_accuracy = exact_matches / len(eval_samples)

print(f"\n" + "=" * 80)
print("📝 WORD-LEVEL SIMILARITY METRICS:")
print("=" * 80)
print(f"   Exact Match Accuracy: {exact_match_accuracy:.4f} ({exact_match_accuracy * 100:.2f}%)")
print(f"   Avg Partial Match:    {avg_partial_match:.4f}")
print(f"   Avg F1 (Word-level):  {avg_f1_word:.4f}")

# Alias for downstream cells
accuracy           = status_accuracy
precision_macro    = status_precision_macro
precision_weighted = status_precision_weighted
recall_macro       = status_recall_macro
recall_weighted    = status_recall_weighted
f1_macro           = status_f1_macro
f1_weighted        = status_f1_weighted
unique_labels      = unique_status_labels
conf_matrix        = status_conf_matrix
y_true             = y_true_status
y_pred             = y_pred_status

print(f"\n✅ Metrics calculated successfully!")

In [ ]:
# Cell 11: Manual inspection of predictions
print("\n" + "=" * 80)
print("🔍 MANUAL INSPECTION OF PREDICTIONS")
print("=" * 80 + "\n")

unknown_count    = sum(1 for r in results if extract_status_label(r["predicted"]) == "UNKNOWN")
normal_count     = sum(1 for r in results if extract_status_label(r["predicted"]) == "NORMAL")
suspicious_count = sum(1 for r in results if extract_status_label(r["predicted"]) == "SUSPICIOUS")

print("=" * 80)
print("⚠️  BASE MODEL OUTPUT ANALYSIS")
print("=" * 80)
print(f"Total predictions:  {len(results):,}")
print(f"  SUSPICIOUS: {suspicious_count:,}  ({suspicious_count / len(results) * 100:.1f}%)")
print(f"  NORMAL:     {normal_count:,}     ({normal_count / len(results) * 100:.1f}%)")
print(f"  UNKNOWN:    {unknown_count:,}     ({unknown_count / len(results) * 100:.1f}%)")
print()

if unknown_count > 0:
    print(f"⚠️  {unknown_count} predictions could not be parsed.")
    print("   Sample unparseable output:")
    for r in results:
        if extract_status_label(r["predicted"]) == "UNKNOWN":
            print(f"   {r['predicted'][:300]}")
            break

# Show a few correct and incorrect predictions
correct   = [r for r in results if extract_status_label(r["predicted"]) == extract_status_label(r["expected"])]
incorrect = [r for r in results if extract_status_label(r["predicted"]) != extract_status_label(r["expected"])]

print(f"\n✅ Correct:   {len(correct):,} / {len(results):,}")
print(f"❌ Incorrect: {len(incorrect):,} / {len(results):,}")

if incorrect:
    print("\n🔎 Sample Incorrect Prediction:")
    r = incorrect[0]
    print(f"   Input:     {r['input'][:120]}...")
    print(f"   Expected:  {r['expected'][:200]}...")
    print(f"   Predicted: {r['predicted'][:200]}...")
    print(f"   True label: {extract_status_label(r['expected'])} | "
          f"Pred label: {extract_status_label(r['predicted'])}")

In [ ]:
# Cell 12: Final summary report
print("\n" + "=" * 80)
print("🎉 FINAL EVALUATION SUMMARY — BASE MODEL")
print("=" * 80 + "\n")

print(f"🤖 Model: {CONFIG['model_name']}  (base — no fine-tuning)")
print(f"⚡ Inference: vLLM offline batch\n")

print(f"📊 Dataset Information:")
print(f"   Total samples evaluated: {len(eval_samples):,}")
print(f"   Inference time:          {elapsed / 60:.2f} minutes")
print(f"   Time per sample:         {elapsed / len(eval_samples):.2f} seconds")

print(f"\n🎯 Key Performance Metrics:")
print(f"   ✓ Overall Accuracy:       {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"   ✓ Precision (Weighted):   {precision_weighted:.4f}")
print(f"   ✓ Recall (Weighted):      {recall_weighted:.4f}")
print(f"   ✓ F1-Score (Weighted):    {f1_weighted:.4f}")
print(f"   ✓ Precision (Macro):      {precision_macro:.4f}")
print(f"   ✓ Recall (Macro):         {recall_macro:.4f}")
print(f"   ✓ F1-Score (Macro):       {f1_macro:.4f}")
print(f"\n   ✓ Exact Match Accuracy:   {exact_match_accuracy:.4f} ({exact_match_accuracy * 100:.2f}%)")
print(f"   ✓ Avg Partial Match:      {avg_partial_match:.4f}")
print(f"   ✓ Avg F1 (Word-level):    {avg_f1_word:.4f}")

print(f"\n{'=' * 80}")
print("✅ Evaluation Complete!")
print("   Compare these numbers against Fine-Tune/metrics.ipynb to see the")
print("   impact of LoRA fine-tuning on MITRE ATT&CK detection.")
print("=" * 80)

In [ ]:
# Cell 13: Save results to CSV
print("💾 Saving detailed results...\n")

import pandas as pd

results_df = pd.DataFrame(results)
results_df["true_label"]      = y_true
results_df["predicted_label"] = y_pred
results_df["correct"]         = results_df["true_label"] == results_df["predicted_label"]

if CONFIG["save_results"]:
    results_df.to_csv(CONFIG["results_file"], index=False)
    print(f"✅ Detailed results saved to: {CONFIG['results_file']}")
else:
    print("ℹ️  Results not saved (CONFIG['save_results'] = False)")

# Metrics summary
metrics_summary = {
    "Metric": [
        "Model",
        "Accuracy",
        "Precision (Macro)",
        "Precision (Weighted)",
        "Recall (Macro)",
        "Recall (Weighted)",
        "F1-Score (Macro)",
        "F1-Score (Weighted)",
        "Exact Match Accuracy",
        "Avg Partial Match",
        "Avg F1 (Word-level)",
    ],
    "Score": [
        CONFIG["model_name"] + " (base)",
        accuracy,
        precision_macro,
        precision_weighted,
        recall_macro,
        recall_weighted,
        f1_macro,
        f1_weighted,
        exact_match_accuracy,
        avg_partial_match,
        avg_f1_word,
    ],
}
metrics_df = pd.DataFrame(metrics_summary)

if CONFIG["save_results"]:
    metrics_df.to_csv(CONFIG["metrics_file"], index=False)
    print(f"✅ Metrics summary saved to: {CONFIG['metrics_file']}")

print("\n📋 Sample Results (First 10):")
display_cols = ["true_label", "predicted_label", "correct", "f1_score"]
print(results_df[display_cols].head(10).to_string(index=False))

print(f"\n📊 Correct:   {results_df['correct'].sum()} / {len(results_df)} ({accuracy * 100:.2f}%)")
print(f"📊 Incorrect: {(~results_df['correct']).sum()} / {len(results_df)} ({(1 - accuracy) * 100:.2f}%)")

In [ ]:
# Cell 14: Visualize metrics
print("🎨 Creating metrics visualization...\n")

import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    f"Base Model Evaluation — {CONFIG['model_name']}",
    fontsize=15, fontweight="bold", y=1.02,
)

# Plot 1: Overall metrics
metrics_names  = ["Accuracy", "Precision\n(Weighted)", "Recall\n(Weighted)", "F1-Score\n(Weighted)"]
metrics_values = [accuracy, precision_weighted, recall_weighted, f1_weighted]

bars1 = ax1.bar(
    metrics_names, metrics_values,
    color=["#2ecc71", "#3498db", "#e74c3c", "#f39c12"], alpha=0.8,
)
ax1.set_ylabel("Score", fontsize=12, fontweight="bold")
ax1.set_title("Overall Performance Metrics", fontsize=14, fontweight="bold")
ax1.set_ylim([0, 1])
ax1.axhline(y=0.5, color="gray", linestyle="--", alpha=0.3, label="50% baseline")
ax1.grid(axis="y", alpha=0.3)
for bar in bars1:
    h = bar.get_height()
    ax1.text(
        bar.get_x() + bar.get_width() / 2., h,
        f"{h:.3f}\n({h * 100:.1f}%)",
        ha="center", va="bottom", fontweight="bold",
    )

# Plot 2: Macro vs Weighted
metrics_comparison = {
    "Precision": [precision_macro, precision_weighted],
    "Recall":    [recall_macro,    recall_weighted],
    "F1-Score":  [f1_macro,        f1_weighted],
}
x = np.arange(len(metrics_comparison))
width = 0.35

bars2_1 = ax2.bar(x - width / 2, [v[0] for v in metrics_comparison.values()],
                  width, label="Macro",    color="#3498db", alpha=0.8)
bars2_2 = ax2.bar(x + width / 2, [v[1] for v in metrics_comparison.values()],
                  width, label="Weighted", color="#e74c3c", alpha=0.8)

ax2.set_ylabel("Score", fontsize=12, fontweight="bold")
ax2.set_title("Macro vs Weighted Metrics", fontsize=14, fontweight="bold")
ax2.set_xticks(x)
ax2.set_xticklabels(metrics_comparison.keys())
ax2.set_ylim([0, 1])
ax2.legend()
ax2.grid(axis="y", alpha=0.3)
for bars in [bars2_1, bars2_2]:
    for bar in bars:
        h = bar.get_height()
        ax2.text(
            bar.get_x() + bar.get_width() / 2., h,
            f"{h:.3f}",
            ha="center", va="bottom", fontsize=9,
        )

plt.tight_layout()
plt.savefig("base_model_metrics.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Metrics visualization saved to base_model_metrics.png")

In [ ]:
# Cell 15: Detailed classification report
print("📊 DETAILED CLASSIFICATION REPORT")
print("=" * 80 + "\n")

import pandas as pd
from sklearn.metrics import classification_report

report    = classification_report(y_true, y_pred, labels=unique_labels, zero_division=0, output_dict=True)
report_df = pd.DataFrame(report).transpose()

print(classification_report(y_true, y_pred, labels=unique_labels, zero_division=0))
print("\n📈 Per-Class Metrics Summary:")
print(report_df.round(4))

if len(unique_labels) > 5:
    class_rows    = report_df[~report_df.index.isin(["accuracy", "macro avg", "weighted avg"])]
    class_metrics = class_rows.sort_values("f1-score", ascending=False)
    print("\n🏆 TOP 5 BEST PERFORMING CLASSES:")
    print(class_metrics.head(5)[["precision", "recall", "f1-score", "support"]].round(4))
    print("\n⚠️  TOP 5 WORST PERFORMING CLASSES:")
    print(class_metrics.tail(5)[["precision", "recall", "f1-score", "support"]].round(4))
elif len(unique_labels) > 0:
    class_rows = report_df[~report_df.index.isin(["accuracy", "macro avg", "weighted avg"])]
    if len(class_rows) > 0:
        print(f"\n📊 Status Classification Summary ({len(unique_labels)} classes):")
        print(class_rows[["precision", "recall", "f1-score", "support"]].round(4))

In [ ]:
# Cell 16: Confusion matrix
print("🎨 Creating confusion matrix visualization...\n")

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

fig, ax = plt.subplots(figsize=(max(8, len(unique_labels) * 2), max(6, len(unique_labels) * 2)))

if len(unique_labels) > 20:
    from collections import Counter
    top_labels    = [lbl for lbl, _ in Counter(y_true).most_common(20)]
    label_indices = [unique_labels.index(lbl) for lbl in top_labels]
    cm_subset     = conf_matrix[np.ix_(label_indices, label_indices)]
    sns.heatmap(cm_subset, annot=True, fmt="d", cmap="Blues",
                xticklabels=top_labels, yticklabels=top_labels, ax=ax)
    ax.set_title(f"Confusion Matrix (Top 20 Labels) — Base Model",
                 fontsize=14, fontweight="bold", pad=20)
else:
    # Normalised annotation: show both count and row-% 
    row_sums = conf_matrix.sum(axis=1, keepdims=True)
    cm_norm  = np.where(row_sums > 0, conf_matrix / row_sums, 0)
    annot    = np.array(
        [[f"{conf_matrix[i,j]}\n({cm_norm[i,j]*100:.1f}%)"
          for j in range(conf_matrix.shape[1])]
         for i in range(conf_matrix.shape[0])]
    )
    sns.heatmap(cm_norm, annot=annot, fmt="", cmap="Blues",
                xticklabels=unique_labels, yticklabels=unique_labels,
                ax=ax, vmin=0, vmax=1)
    ax.set_title("Confusion Matrix — Base Model (row-normalised)",
                 fontsize=14, fontweight="bold", pad=20)

ax.set_xlabel("Predicted Label", fontsize=12, fontweight="bold")
ax.set_ylabel("True Label",      fontsize=12, fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("base_model_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Confusion matrix saved to base_model_confusion_matrix.png")